# Tâche #3 : Classification d'incidents avec des modèles *Transformers*

On reprend le thème de la classification de descriptions d’incidents. Utilisez la librairie *HuggingFace* pour accomplir cette tâche. On vous demande plus spécifiquement d’utiliser 2 modèles : le modèle *bert-base-uncased* et un autre modèle de votre choix. Débutez votre travail à partir de ce notebook. 

Le corpus de textes contient 3 partitions : 

-	*data/incidents_train.json* : fichier d’entraînement 
-	*data/incidents_dev.json* : fichier de validation
-	*data/incidents_test.json* : fichier de test 
 
Vous pouvez ajouter au *notebook* toutes les cellules dont vous avez besoin pour votre code, vos explications ou la présentation de vos résultats. Vous pouvez également ajouter des sous-sections (par ex. des sous-sections 4.1, 4.2 etc.) si cela améliore la lisibilité.

## 1. Création du jeu de données (*les 3 partitions du dataset*)

In [1]:
import json

train_json_fn = "./data/incidents_train.json"
dev_json_fn = "./data/incidents_dev.json"
test_json_fn = "./data/incidents_test.json"


def load_json_data(filename):
    with open(filename, 'r') as fp:
        data = json.load(fp)
    return data

In [2]:
print("\n==== Entrainement")
train_list = load_json_data(train_json_fn)
print("Nombre d'incidents:", len(train_list))
print("Un exemple:\n", train_list[10])



==== Entrainement
Nombre d'incidents: 2475
Un exemple:
 {'text': " At approximately 12:15 p.m. on February 8  2011  Employee # 1 and four other  coworkers were working in the West storage Facility at Enterprise Products  located at Mont Belvieu  Texas. A major fire began but Employee # 1 wasn't  able to escape. The recovery effort was delayed until the evening of February  10  2011  due to the presence of hydrocarbon vapors and small residual fires.   ", 'label': '6'}


In [3]:
print("\n==== Validation")
dev_list = load_json_data(dev_json_fn)
print("Nombre d'incidents:", len(dev_list))
print("Un exemple:\n", dev_list[10])


==== Validation
Nombre d'incidents: 531
Un exemple:
 {'text': " On November 30  2010 an employee had suffered a serious injury. The Employer  was an underground utility contractor that was installing a storm water gate.  The injured employee was a laborer with over 9 years'of experience with the  employer. On the day of the incident  two components of the system were being  picked at the same time to place into position. The top section began to slide  and fall off as the load shifted. The injured employee  who was guiding the  stacked parts to its delivery point  jumped away to avoid the falling  prefabricated concrete part as it fell  and suffered rib and back injury that  required hospitalization for two days. The falling concrete part weighed in  excess of 2000 pounds.                                                          ", 'label': '5'}


In [4]:
print("\n==== Test")
test_list = load_json_data(test_json_fn)
print("Nombre d'incidents:", len(test_list))
print("Un exemple:\n", test_list[10])


==== Test
Nombre d'incidents: 531
Un exemple:
 {'text': ' On May 24  2004  Employee #1 was welding the deck caps on the of fourth floor  decking. The employee fell from the edge and sustained life threatening head  and internal injuries. Employee #1 was then transferred to the Kaweah Delta  District Hospital  where he subsequently died from his injuries. The employee  was not wearing any fall protection but there was a single line of wire rope  at the perimeter of the structure.                                              ', 'label': '5'}


In [5]:
# création du dataset conforme au dataset de HuggingFace
print("\n==== Dataset de HuggingFace")

from datasets import Dataset, DatasetDict


def to_dataset(examples):
    return Dataset.from_dict({
        "text":   [ex["text"].strip() for ex in examples],
        "labels": [int(ex["label"]) for ex in examples],
    })


dataset = DatasetDict({
    "train":      to_dataset(train_list),
    "validation": to_dataset(dev_list),
    "test":       to_dataset(test_list),
})

num_labels = len(set(dataset["train"]["labels"]))
print(dataset)
print("Nombre de classes :", num_labels)


==== Dataset de HuggingFace


c:\Users\valen\Desktop\IFT_7022_TP1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 2475
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 531
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 531
    })
})
Nombre de classes : 9


## 2. Création des 2 modèles

In [6]:
# Tokenization du dataset
print("\n==== Tokenization du dataset")

import numpy as np
from transformers import AutoTokenizer

BERT_BASE_UNCASED = "google-bert/bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(BERT_BASE_UNCASED)

# Longueur des textes en jetons BERT (pour choisir MAX_LENGTH)
lengths = np.array([len(bert_tokenizer(t)["input_ids"]) for t in dataset["train"]["text"]])
print("Longueur moyenne :", round(lengths.mean()))
print("Médiane          :", int(np.median(lengths)))
print("95e centile      :", int(np.percentile(lengths, 95)))
print("Maximum          :", lengths.max())
print("Textes > 512 jetons :", round((lengths > 512).mean() * 100, 1), "%")



MAX_LENGTH = 512   # limite de BERT ; à ajuster selon les statistiques ci-dessus

def tokenize_bert(batch):
    """ Tokenise un lot de textes ; les textes trop longs sont tronqués à MAX_LENGTH jetons. """
    return bert_tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


bert_dataset = dataset.map(tokenize_bert, batched=True)
print(bert_dataset)


==== Tokenization du dataset


c:\Users\valen\Desktop\IFT_7022_TP1\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (623 > 512). Running this sequence through the model will result in indexing errors


Longueur moyenne : 147
Médiane          : 119
95e centile      : 342
Maximum          : 850
Textes > 512 jetons : 1.2 %


Map: 100%|██████████| 531/531 [00:00<00:00, 7145.34 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2475
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 531
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 531
    })
})


### 2.1 Modèle BERT


In [ ]:
# Creation du model BERT
print("\n==== Modèle BERT")

from transformers import AutoModelForSequenceClassification

bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_BASE_UNCASED,
    num_labels=num_labels,
)
print(bert_model)


==== Modèle BERT


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

### 2.2 Deuxième modèle


In [7]:
# 2.2 Deuxième modèle : ELECTRA
print("\n==== Modèle ELECTRA")

from transformers import AutoTokenizer, AutoModelForSequenceClassification

ELECTRA_MODEL = "google/electra-base-discriminator"


electra_tokenizer = AutoTokenizer.from_pretrained(ELECTRA_MODEL)


def tokenize_electra(batch):
    return electra_tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


electra_dataset = dataset.map(tokenize_electra, batched=True)


electra_model = AutoModelForSequenceClassification.from_pretrained(
    ELECTRA_MODEL,
    num_labels=num_labels,
)
print(electra_model)


==== Modèle ELECTRA


Map: 100%|██████████| 531/531 [00:00<00:00, 6529.68 examples/s]
Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at google/electra-base-discriminator and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

## 3. Entraînement des 2 modèles

### 3.1 Modèle BERT


In [35]:
# Padding
print("\n==== Padding")

import numpy as np
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score

# Padding batch par batch à la longueur du plus long texte du lot
dataCollator = DataCollatorWithPadding(tokenizer=bert_tokenizer)

# definition des metrique d'evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}


==== Padding


In [36]:
# Hyperparamètres
print("\n==== Hyperparamètres")

from transformers import TrainingArguments

bert_args = TrainingArguments(
    output_dir="bert_incidents",
    learning_rate=2e-5,                 # valeur classique pour l'affinage de BERT
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",              # évaluation sur la validation à chaque époque
    save_strategy="epoch",
    load_best_model_at_end=True,        # garde la meilleure époque
    metric_for_best_model="accuracy",
    save_total_limit=1,
    logging_steps=50,
    seed=42,
    report_to="none",
)


==== Hyperparamètres


In [38]:
# Entraînement
print("\n==== Entraînement de BERT")

from transformers import Trainer

bert_trainer = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=bert_dataset["train"],
    eval_dataset=bert_dataset["validation"],
    processing_class=bert_tokenizer,
    data_collator=dataCollator,
    compute_metrics=compute_metrics,
)

bert_trainer.train()


==== Entraînement de BERT


/Users/macbookpro/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.153691,1.263731,0.783427
2,0.165203,1.239080,0.785311
3,0.062555,1.322115,0.790960
4,0.025027,1.401682,0.806026
5,0.048879,1.392859,0.807910
6,0.004486,1.397002,0.806026


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/macbookpro/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/macbookpro/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=1860, training_loss=0.05852620603156186, metrics={'train_runtime': 944.118, 'train_samples_per_second': 15.729, 'train_steps_per_second': 1.97, 'total_flos': 2395532612065230.0, 'train_loss': 0.05852620603156186, 'epoch': 6.0})

### 3.2 Deuxième modèle


In [10]:
# 3.2 Entraînement de ELECTRA
print("\n==== Entraînement de ELECTRA")
import torch
print("GPU NVIDIA détecté et activé :", torch.cuda.is_available())
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

from transformers import DataCollatorWithPadding, TrainingArguments, Trainer


electra_collator = DataCollatorWithPadding(tokenizer=electra_tokenizer)


electra_args = TrainingArguments(
    output_dir="electra_incidents",
    learning_rate=2e-5,
    num_train_epochs=6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,
    logging_steps=50,
    seed=42,
    report_to="none",
    save_safetensors=False,
)


electra_trainer = Trainer(
    model=electra_model,
    args=electra_args,
    train_dataset=electra_dataset["train"],
    eval_dataset=electra_dataset["validation"],
    tokenizer=electra_tokenizer,
    data_collator=electra_collator,
    compute_metrics=compute_metrics,
)


electra_trainer.train()


==== Entraînement de ELECTRA
GPU NVIDIA détecté et activé : True


                                                  
 17%|█▋        | 310/1860 [51:51<24:32,  1.05it/s] 

{'loss': 1.0084, 'grad_norm': 15.888144493103027, 'learning_rate': 1.946236559139785e-05, 'epoch': 0.16}


                                                  
 17%|█▋        | 310/1860 [53:15<24:32,  1.05it/s]

{'loss': 0.9019, 'grad_norm': 10.658652305603027, 'learning_rate': 1.89247311827957e-05, 'epoch': 0.32}


...



































                                                    
                                                  

 17%|█▋        | 310/1860 [3:15:38<24:32,  1.05it/s]



{'eval_loss': 0.8931204676628113, 'eval_accuracy': 0.8173258003766478, 'eval_runtime': 38.7678, 'eval_samples_per_second': 13.697, 'eval_steps_per_second': 0.877, 'epoch': 6.0}


                                                    
100%|██████████| 1860/1860 [2:25:37<00:00,  4.70s/it]

{'train_runtime': 8737.5183, 'train_samples_per_second': 1.7, 'train_steps_per_second': 0.213, 'train_loss': 0.43701522427220496, 'epoch': 6.0}


TrainOutput(global_step=1860, training_loss=0.43701522427220496, metrics={'train_runtime': 8737.5183, 'train_samples_per_second': 1.7, 'train_steps_per_second': 0.213, 'total_flos': 2395532612065230.0, 'train_loss': 0.43701522427220496, 'epoch': 6.0})

## 4. Évaluation et résultats

In [43]:
# Évaluation de BERT sur le test
print("\n==== Évaluation de BERT sur le test")

from sklearn.metrics import classification_report

bert_test_output = bert_trainer.predict(bert_dataset["test"])
bert_test_predictions = np.argmax(bert_test_output.predictions, axis=-1)
bert_test_labels = bert_test_output.label_ids

print("Accuracy :", round(accuracy_score(bert_test_labels, bert_test_predictions), 3))
# Évaluation de BERT sur le test
print("\n==== Évaluation de BERT sur le test")

bert_test_results = bert_trainer.evaluate(bert_dataset["test"])



==== Évaluation de BERT sur le test


/Users/macbookpro/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Accuracy : 0.761

==== Évaluation de BERT sur le test


/Users/macbookpro/Library/Python/3.10/lib/python/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.004486,1.843844,6,0.760829


In [11]:
# Évaluation de ELECTRA sur le test
print("\n==== Évaluation de ELECTRA sur le test")

# Obtenir les prédictions
electra_test_output = electra_trainer.predict(electra_dataset["test"])
electra_test_predictions = np.argmax(electra_test_output.predictions, axis=-1)
electra_test_labels = electra_test_output.label_ids

# Afficher l'accuracy
from sklearn.metrics import accuracy_score
print("Accuracy ELECTRA :", round(accuracy_score(electra_test_labels, electra_test_predictions), 3))

# Afficher les métriques complètes de HuggingFace
electra_test_results = electra_trainer.evaluate(electra_dataset["test"])
print(electra_test_results)


==== Évaluation de ELECTRA sur le test


100%|██████████| 34/34 [00:52<00:00,  1.53s/it]


Accuracy ELECTRA : 0.785


100%|██████████| 34/34 [00:50<00:00,  1.49s/it]

{'eval_loss': 0.7941817045211792, 'eval_accuracy': 0.7853107344632768, 'eval_runtime': 52.1906, 'eval_samples_per_second': 10.174, 'eval_steps_per_second': 0.651, 'epoch': 6.0}


In [12]:
import pandas as pd
from IPython.display import display

df_results = pd.DataFrame({
    "Métrique": ["Exactitude (Accuracy)", "Perte (Loss)", "Temps d'évaluation (s)", "Échantillons/sec"],
    "Valeur": [
        f"{electra_test_results['eval_accuracy']:.2%}", 
        round(electra_test_results['eval_loss'], 4), 
        round(electra_test_results['eval_runtime'], 2), 
        round(electra_test_results['eval_samples_per_second'], 2)
    ]
})

# Affichage esthétique sans l'index numérique
display(df_results.style.hide(axis="index"))

Métrique,Valeur
Exactitude (Accuracy),78.53%
Perte (Loss),0.794200
Temps d'évaluation (s),52.190000
Échantillons/sec,10.170000
